# LeetCode 3971 — Maximum Total Value bằng Python

- Link: https://leetcode.com/problems/maximum-total-value/
- Độ khó: **Hard**
- Chủ đề: Tham lam (greedy), tìm kiếm nhị phân trên đáp án (ngưỡng), cấp số cộng, heap, số học lớn / modulo

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxTotalValue(self, value: list[int], decay: list[int], m: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Có `n` "chỉ số". Chọn chỉ số `i` lần thứ `t` (đếm từ 1) cho giá trị `value[i] − decay[i]·(t−1)`: giá trị giảm dần đều mỗi lần chọn. Ta được chọn tổng cộng **tối đa `m` lần** (có thể chọn một chỉ số nhiều lần), muốn **tổng giá trị lớn nhất**, in ra `mod 10^9 + 7`.

Với `m` lên tới `10^9`, ta không thể mô phỏng từng lần chọn. Điểm mấu chốt: mỗi chỉ số sinh ra một **cấp số cộng giảm** các "lượt giá trị", bài toán chính là **lấy `m` số lớn nhất (dương) trong hợp của `n` cấp số cộng** — và ta tìm được "ngưỡng" của số lớn thứ `m` bằng tìm kiếm nhị phân, thay vì sinh từng số.

Sau notebook này bạn sẽ:

1. Phát biểu lại bài toán thành "lấy `m` số lớn nhất trong nhiều dãy giảm" và hiểu vì sao tham lam đúng.
2. Đếm nhanh có bao nhiêu số `≥ t` trong một cấp số cộng, và tính tổng cấp số cộng bằng công thức.
3. Dùng **tìm kiếm nhị phân trên đáp án** (ngưỡng `t`) và xử lý các giá trị **bằng ngưỡng** (hòa).
4. Viết 4 lời giải (vét cạn mọi cách chia số lần, heap mô phỏng, ngưỡng nhị phân, và dạng mở rộng) rồi đối chiếu bằng test ngẫu nhiên.
5. Phân tích `O(n log V)` và chú ý số lớn (tổng có thể vượt `2^63`).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho hai mảng `value`, `decay` (độ dài `n`) và số nguyên `m`. Mỗi lần chọn chỉ số `i`, nếu đây là lần chọn thứ `t` của `i` thì ta nhận `value[i] − decay[i]·(t−1)` (có thể âm). Tổng số lần chọn không quá `m`. Tìm tổng lớn nhất, trả về `mod 10^9 + 7`.

**Ví dụ 1:** `value = [6,5,4]`, `decay = [2,1,1]`, `m = 4` → `19` (chọn 6, 5, 4 và `6−2 = 4`).

**Ví dụ 2:** `value = [7,2,2]`, `decay = [3,2,1]`, `m = 2` → `11` (7 rồi `7−3 = 4`).

**Ví dụ 3:** `value = [4,3]`, `decay = [5,4]`, `m = 5` → `7` (chỉ chọn 4 và 3; mọi lần chọn tiếp theo cho giá trị âm nên bỏ).

**Ràng buộc**

- `1 <= value.length == decay.length <= 10^5`
- `1 <= value[i], decay[i] <= 10^9`
- `1 <= m <= 10^9`

Lưu ý: "tối đa `m` lần" — ta **không bắt buộc** dùng hết `m` lần; không chọn thêm khi giá trị chỉ còn `<= 0`.

In [1]:
examples = [
    (dict(value=[6, 5, 4], decay=[2, 1, 1], m=4), 19),
    (dict(value=[7, 2, 2], decay=[3, 2, 1], m=2), 11),
    (dict(value=[4, 3], decay=[5, 4], m=5), 7),
]
for args, expected in examples:
    print("value =", args["value"], "| decay =", args["decay"], "| m =", args["m"], "| mong đợi:", expected)

value = [6, 5, 4] | decay = [2, 1, 1] | m = 4 | mong đợi: 19
value = [7, 2, 2] | decay = [3, 2, 1] | m = 2 | mong đợi: 11
value = [4, 3] | decay = [5, 4] | m = 5 | mong đợi: 7


## 3. Kiến thức nền

### 3.1. Mỗi chỉ số là một cấp số cộng giảm

Với chỉ số `i`, các lần chọn lần lượt cho `v, v − d, v − 2d, ...` (với `v = value[i]`, `d = decay[i]`). Đây là cấp số cộng công sai `−d`. Chỉ các số **dương** mới đáng chọn.

In [2]:
def gains(v, d):
    """Các giá trị dương của chỉ số có value=v, decay=d."""
    out, g = [], v
    while g > 0:
        out.append(g)
        g -= d
    return out

print(gains(7, 3))     # [7, 4, 1]
print(gains(4, 5))     # [4]
print(gains(10, 10))   # [10]

[7, 4, 1]
[4]
[10]


### 3.2. Đếm số hạng `>= t` trong cấp số cộng bằng chia nguyên

Số hạng thứ `j` (đếm từ `0`) là `v − j·d`. Điều kiện `v − j·d ≥ t` ⇔ `j ≤ (v − t)/d`. Với `v ≥ t`, số hạng thỏa là `j = 0, 1, ..., ⌊(v − t)/d⌋`, tức **`⌊(v − t)/d⌋ + 1`** số hạng. Nếu `v < t` thì không có số hạng nào.

In [3]:
v, d = 7, 3                              # 7, 4, 1
for t in [8, 7, 5, 4, 2, 1]:
    c = (v - t) // d + 1 if v >= t else 0
    print(f"t={t}: có {c} số hạng >= t, thực tế:", [g for g in gains(v, d) if g >= t])

t=8: có 0 số hạng >= t, thực tế: []
t=7: có 1 số hạng >= t, thực tế: [7]
t=5: có 1 số hạng >= t, thực tế: [7]
t=4: có 2 số hạng >= t, thực tế: [7, 4]
t=2: có 2 số hạng >= t, thực tế: [7, 4]
t=1: có 3 số hạng >= t, thực tế: [7, 4, 1]


### 3.3. Tổng của `c` số hạng đầu của cấp số cộng

`v + (v−d) + ... + (v−(c−1)d) = c·v − d·(0 + 1 + ... + (c−1)) = c·v − d·c·(c−1)/2`. Công thức này chạy `O(1)` cho `c` tới `10^9`, nhưng giá trị có thể lên tới `~ 10^18`, và tổng qua `10^5` chỉ số lên tới `~ 10^23` — Python có số nguyên lớn nên không tràn.

In [4]:
def arith_sum(v, d, c):
    return c * v - d * c * (c - 1) // 2

print(arith_sum(7, 3, 3), sum(gains(7, 3)))               # 12 12
print(arith_sum(10**9, 1, 10**9))                          # ~5e17, tính tức thì
print(sum(arith_sum(10**9, 1, 10**9) for _ in range(10**5)) > 2**63)   # True: vượt int64

12 12
500000000500000000
True


### 3.4. Heap để lấy phần tử lớn nhất (max-heap bằng số âm)

`heapq` là min-heap; để lấy **lớn nhất** ta đẩy `−giá_trị`. Phần tử là tuple `(−gain, i)` để biết lượt này thuộc chỉ số nào.

In [5]:
import heapq
h = []
for g, i in [(6, 0), (5, 1), (4, 2)]:
    heapq.heappush(h, (-g, i))
ng, i = heapq.heappop(h)
print("lớn nhất:", -ng, "thuộc chỉ số", i)         # 6 thuộc chỉ số 0

lớn nhất: 6 thuộc chỉ số 0


### 3.5. Tìm kiếm nhị phân trên đáp án (ngưỡng)

Nếu một hàm `f(t)` **đơn điệu** (ở đây `count(t)` không tăng khi `t` tăng), ta có thể tìm giá trị `t` lớn nhất thỏa `f(t) ≥ m` bằng nhị phân thay vì thử từng `t`. Mẫu:

```python
lo, hi = đúng_chắc_chắn, sai_chắc_chắn      # f(lo) đúng, f(hi) sai
while hi - lo > 1:
    mid = (lo + hi) // 2
    if f(mid): lo = mid
    else: hi = mid
# lo là giá trị lớn nhất còn đúng
```

In [6]:
# ví dụ nhỏ: tìm số nguyên lớn nhất t có t*t <= 50
lo, hi = 0, 100         # 0*0 <= 50 đúng; 100*100 <= 50 sai
while hi - lo > 1:
    mid = (lo + hi) // 2
    if mid * mid <= 50:
        lo = mid
    else:
        hi = mid
print(lo)               # 7

7


## 4. Ý tưởng nền tảng / trực giác

Liệt kê mọi "lượt giá trị" dương của ví dụ 1:

| chỉ số | `value`, `decay` | các giá trị dương |
|---|---|---|
| 0 | 6, 2 | 6, 4, 2 |
| 1 | 5, 1 | 5, 4, 3, 2, 1 |
| 2 | 4, 1 | 4, 3, 2, 1 |

Muốn tổng lớn nhất với tối đa `m = 4` lần chọn: chọn 4 giá trị **lớn nhất** trong hợp các dãy: `6, 5, 4, 4` → `19`. Ta dễ thấy hai điều:

1. **Tính "tiền tố":** trong mỗi dãy, nếu ta chọn giá trị thứ `j` thì ta nên chọn mọi giá trị trước nó (lớn hơn). Tập `m` số lớn nhất, nếu hòa thì chọn các số đầu dãy trước, luôn thỏa tính chất này. Nên mọi tập "m số lớn nhất" **hiện thực hóa được** bằng thứ tự chọn hợp lệ (lần thứ `t` của chỉ số `i` phải sau lần `t−1`).
2. **Chỉ lấy số dương:** số `≤ 0` không làm tổng tăng; vì "tối đa `m`" ta bỏ chúng.

Vậy bài toán là: **tổng của `m` số dương lớn nhất** (hoặc ít hơn nếu hết số dương). Với `m` lớn ta không liệt kê các số; ta tìm "số lớn thứ `m`".

## 5. Cách 1 — Vét cạn: thử mọi cách chia số lần chọn cho từng chỉ số

**Ý tưởng.** Gọi `c_i` là số lần chọn chỉ số `i`, với `Σ c_i ≤ m`. Tổng giá trị là `Σ (c_i·v_i − d_i·c_i(c_i−1)/2)`. Đệ quy thử mọi `c_i` từ `0` tới số lần còn lại. Cách này **không** dùng tham lam nên là một "thước đo" độc lập đúng theo định nghĩa đề.

In [7]:
MOD = 10**9 + 7

def brute_force(value, decay, m):
    n = len(value)
    best = [0]

    def go(i, left, acc):
        if i == n:
            best[0] = max(best[0], acc)
            return
        for c in range(left + 1):
            go(i + 1, left - c, acc + c * value[i] - decay[i] * c * (c - 1) // 2)

    go(0, m, 0)
    return best[0]                 # giá trị chính xác (chưa mod) để so sánh chặt chẽ

for args, expected in examples:
    got = brute_force(**args)
    print(got, expected)
    assert got % MOD == expected

19 19
11 11
7 7


**Độ phức tạp.** Số cách chia `m` lần cho `n` chỉ số là `C(m + n, n)` — bùng nổ. Chỉ dùng cho `n, m` rất nhỏ.

**Nút thắt.** Việc phân chia số lần cho từng chỉ số thật ra là hệ quả của việc lấy `m` giá trị lớn nhất (phần 4); ta không cần thử mọi phân chia.

## 6. Bước đột phá

**Insight 1 — bài toán = tổng `m` số dương lớn nhất.** Như đã nêu: mỗi chỉ số là một dãy giảm; chọn `m` số lớn nhất.

**Insight 2 — "ngưỡng" `t`.** Gọi `count(t)` = số lượt giá trị `≥ t` (qua mọi chỉ số) và `S(t)` = tổng các lượt giá trị `≥ t`. `count(t)` **không tăng** khi `t` tăng.

- Nếu `count(1) ≤ m` (số lượt dương ít hơn hoặc bằng `m`) thì lấy **tất cả**: đáp án `S(1)`.
- Ngược lại, tồn tại ngưỡng lớn nhất `t*` với `count(t*) ≥ m` (và `count(t*+1) < m`). Mọi lượt `≥ t*+1` đều được chọn (có `count(t*+1) < m` lượt), phần còn thiếu `m − count(t*+1)` lượt lấy từ các lượt có giá trị **đúng bằng `t*`** (chắc chắn đủ, vì `count(t*) ≥ m`).

**Công thức:** `đáp án = S(t* + 1) + (m − count(t* + 1)) · t*`.

**Insight 3 — tính `count` và `S` trong `O(n)`.** Mỗi chỉ số một công thức: `c = (v − t)//d + 1` (nếu `v ≥ t`), tổng `c·v − d·c(c−1)/2`. Nhị phân trên `t ∈ [1, max(value)]` tốn `log2(10^9) ≈ 30` lần ⇒ `O(30 · n)`.

**Insight 4 — mod chỉ ở cuối.** Tổng chính xác là số nguyên lớn; Python tính được, ta chỉ `% MOD` ở bước cuối (hoặc mod từng phần cẩn thận với phép chia cho `2`).

**Quy tắc "nếu… thì…":**

- Nếu `value[i] < t` thì chỉ số `i` không có lượt `≥ t`.
- Nếu `count(1) ≤ m` thì không cần nhị phân (lấy hết lượt dương).
- Nếu nhiều lượt hòa đúng bằng `t*`, chỉ cần số lượng `m − count(t*+1)`, không cần biết là của chỉ số nào (chúng cùng giá trị `t*`).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mô phỏng bằng heap (greedy theo từng lần chọn)

**Giải thích từng đoạn.**

1. Đẩy lượt đầu tiên `value[i]` của mỗi chỉ số vào max-heap (bằng số âm).
2. Lặp tối đa `m` lần: lấy lượt lớn nhất `g`; nếu `g ≤ 0` dừng (mọi lượt còn lại không dương); cộng `g` vào tổng; đẩy lượt kế tiếp của cùng chỉ số `g − decay[i]`.

Đây là "gộp `n` dãy đã sắp xếp" cổ điển. **Đúng** nhưng chỉ chạy được khi `m` nhỏ.

In [8]:
import heapq

def heap_greedy(value, decay, m):
    h = [(-v, i) for i, v in enumerate(value)]
    heapq.heapify(h)
    total = 0
    for _ in range(m):
        if not h:
            break
        ng, i = heapq.heappop(h)
        g = -ng
        if g <= 0:
            break
        total += g
        heapq.heappush(h, (-(g - decay[i]), i))
    return total

for args, expected in examples:
    got = heap_greedy(**args)
    print(got, expected)
    assert got % MOD == expected

19 19
11 11
7 7


**Độ phức tạp.** `O((n + m) log n)` — với `m = 10^9` là bất khả thi. Đây là thước đo "đúng" cho `m` vừa phải (kiểm chứng cách 3).

### 7.2. Cách 3 — Nhị phân trên ngưỡng (bản chính, `O(n log V)`)

**Giải thích từng đoạn.**

1. `count(t)`: với mỗi chỉ số có `v ≥ t` cộng `(v − t)//d + 1`.
2. `total_ge(t)`: với mỗi chỉ số có `v ≥ t`, `c = (v − t)//d + 1`, cộng `c·v − d·c(c−1)//2`.
3. Nếu `count(1) ≤ m` trả `total_ge(1)`.
4. Ngược lại nhị phân tìm `t*` lớn nhất với `count(t*) ≥ m`: `lo = 1` (đúng), `hi = max(value) + 1` (`count = 0 < m`, sai).
5. `ans = total_ge(t* + 1) + (m − count(t* + 1)) · t*`.

In [9]:
def count_ge(value, decay, t):
    return sum((v - t) // d + 1 for v, d in zip(value, decay) if v >= t)

def total_ge(value, decay, t):
    s = 0
    for v, d in zip(value, decay):
        if v >= t:
            c = (v - t) // d + 1
            s += c * v - d * c * (c - 1) // 2
    return s

def threshold_search(value, decay, m):
    if count_ge(value, decay, 1) <= m:               # lấy mọi lượt dương
        return total_ge(value, decay, 1)
    lo, hi = 1, max(value) + 1                       # count(lo) >= m (vì > m), count(hi) = 0 < m
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if count_ge(value, decay, mid) >= m:
            lo = mid
        else:
            hi = mid
    t = lo
    c_above = count_ge(value, decay, t + 1)
    return total_ge(value, decay, t + 1) + (m - c_above) * t

for args, expected in examples:
    got = threshold_search(**args)
    print(got, expected)
    assert got % MOD == expected

19 19
11 11
7 7


**Độ phức tạp.** Mỗi lần gọi `count`/`total` tốn `O(n)`; nhị phân `≈ 30` lần ⇒ `O(n log V)` với `V = max(value) ≤ 10^9`. Bộ nhớ `O(1)` thêm.

### 7.3. Cách 4 — Vét cạn "đúng m lần" để thấy vai trò của "tối đa"

Để hiểu vì sao phải bỏ các lượt `≤ 0`, ta viết biến thể ép **đúng `m` lần** bằng cùng ngưỡng nhưng cho phép `t` âm. Trên input của đề, đáp án "đúng `m`" ≤ "tối đa `m`" và bằng nhau khi mọi lượt thứ `m` vẫn dương.

Với ví dụ 3 (`value = [4,3]`, `decay = [5,4]`, `m = 5`): ép đúng 5 lần phải chọn thêm `4−5 = −1`, `3−4 = −1`, `4−10 = −6`... nên tổng nhỏ hơn `7`.

In [10]:
def exact_m_brute(value, decay, m):
    n = len(value)
    best = [None]
    def go(i, left, acc):
        if i == n:
            if left == 0 and (best[0] is None or acc > best[0]):
                best[0] = acc
            return
        for c in range(left + 1):
            go(i + 1, left - c, acc + c * value[i] - decay[i] * c * (c - 1) // 2)
    go(0, m, 0)
    return best[0]

print("tối đa 5 lần:", brute_force([4, 3], [5, 4], 5), "| đúng 5 lần:", exact_m_brute([4, 3], [5, 4], 5))
print("tối đa 2 lần:", brute_force([4, 3], [5, 4], 2), "| đúng 2 lần:", exact_m_brute([4, 3], [5, 4], 2))

tối đa 5 lần: 7 | đúng 5 lần: 0
tối đa 2 lần: 7 | đúng 2 lần: 7


## 8. TRACE từng bước

Ta trace `threshold_search` cho ví dụ 1 (`value = [6,5,4]`, `decay = [2,1,1]`, `m = 4`). In mỗi bước nhị phân: `[lo, hi)`, `mid`, `count(mid)`, và quyết định; cuối cùng ráp công thức.

Bảng mong đợi:

| `mid` | `count(mid)` | so với `m = 4` | cập nhật |
|---|---|---|---|
| 4 (`lo=1, hi=7`) | `2 + 2 + 1 = 5` | `≥ 4` | `lo = 4` |
| 5 (`lo=4, hi=7`) | `1 + 1 + 0 = 2` | `< 4` | `hi = 5` |
| (dừng, `hi − lo = 1`) | | | `t* = 4` |

Ráp: `count(5) = 2`, `S(5) = 6 + 5 = 11`; thiếu `4 − 2 = 2` lượt giá trị đúng `4` ⇒ `11 + 2·4 = 19` ✓.

In [11]:
def trace_threshold(value, decay, m):
    c1 = count_ge(value, decay, 1)
    print(f"count(1) = {c1} (m = {m})")
    if c1 <= m:
        print("lấy tất cả lượt dương:", total_ge(value, decay, 1)); return
    lo, hi = 1, max(value) + 1
    step = 0
    while hi - lo > 1:
        mid = (lo + hi) // 2
        c = count_ge(value, decay, mid)
        step += 1
        verdict = "count >= m -> lo = mid" if c >= m else "count < m -> hi = mid"
        print(f"bước {step}: lo={lo}, hi={hi}, mid={mid}, count(mid)={c} => {verdict}")
        if c >= m: lo = mid
        else: hi = mid
    t = lo
    ca = count_ge(value, decay, t + 1)
    sa = total_ge(value, decay, t + 1)
    print(f"ngưỡng t* = {t}; count(t*+1) = {ca}; S(t*+1) = {sa}")
    ans = sa + (m - ca) * t
    print(f"đáp án = {sa} + ({m} - {ca}) * {t} = {ans}")
    return ans

trace_threshold(**examples[0][0])

count(1) = 12 (m = 4)
bước 1: lo=1, hi=7, mid=4, count(mid)=5 => count >= m -> lo = mid
bước 2: lo=4, hi=7, mid=5, count(mid)=2 => count < m -> hi = mid
ngưỡng t* = 4; count(t*+1) = 2; S(t*+1) = 11
đáp án = 11 + (4 - 2) * 4 = 19


19

In [12]:
# Trace thêm: ví dụ 2 (m nhỏ, có hòa) và ví dụ 3 (hết lượt dương trước khi hết m)
print("--- ví dụ 2 ---"); trace_threshold(**examples[1][0])
print("--- ví dụ 3 ---"); trace_threshold(**examples[2][0])
print("--- liệt kê tay các lượt của ví dụ 2 ---")
for i, (v, d) in enumerate(zip([7, 2, 2], [3, 2, 1])):
    print(i, gains(v, d))

--- ví dụ 2 ---
count(1) = 6 (m = 2)
bước 1: lo=1, hi=8, mid=4, count(mid)=2 => count >= m -> lo = mid
bước 2: lo=4, hi=8, mid=6, count(mid)=1 => count < m -> hi = mid
bước 3: lo=4, hi=6, mid=5, count(mid)=1 => count < m -> hi = mid
ngưỡng t* = 4; count(t*+1) = 1; S(t*+1) = 7
đáp án = 7 + (2 - 1) * 4 = 11
--- ví dụ 3 ---
count(1) = 2 (m = 5)
lấy tất cả lượt dương: 7
--- liệt kê tay các lượt của ví dụ 2 ---
0 [7, 4, 1]
1 [2]
2 [2, 1]


Ở ví dụ 3, `count(1) = 2 ≤ m = 5` nên ta thoát sớm và lấy mọi lượt dương: `4 + 3 = 7`. Ở ví dụ 2, ngưỡng `t* = 4`: các lượt `≥ 5` chỉ có `7` (1 lượt); thiếu `2 − 1 = 1` lượt giá trị đúng `4` ⇒ `7 + 4 = 11`.

## 9. Code nộp LeetCode

Dùng cách 3, gộp `count` và `sum` thành một hàm phụ trong phương thức. Chú ý `% MOD` chỉ ở cuối (Python xử lý số lớn).

In [13]:
class Solution:
    def maxTotalValue(self, value: list[int], decay: list[int], m: int) -> int:
        MOD = 10**9 + 7

        def stat(t):
            """(số lượt >= t, tổng các lượt >= t) trên mọi chỉ số."""
            cnt = tot = 0
            for v, d in zip(value, decay):
                if v >= t:
                    c = (v - t) // d + 1
                    cnt += c
                    tot += c * v - d * c * (c - 1) // 2
            return cnt, tot

        cnt1, tot1 = stat(1)
        if cnt1 <= m:                         # ít lượt dương hơn m: lấy hết
            return tot1 % MOD
        lo, hi = 1, max(value) + 1            # count(lo) > m, count(hi) = 0
        while hi - lo > 1:
            mid = (lo + hi) // 2
            if stat(mid)[0] >= m:
                lo = mid
            else:
                hi = mid
        cnt, tot = stat(lo + 1)               # mọi lượt >= lo+1 đều được chọn
        return (tot + (m - cnt) * lo) % MOD   # phần thiếu lấy từ các lượt đúng bằng lo


sol = Solution()
for args, expected in examples:
    got = sol.maxTotalValue(**args)
    print(got, expected)
    assert got == expected

19 19
11 11
7 7


## 10. Vì sao thuật toán đúng

**Tham lam đúng.** Với một lần chọn tại chỉ số `i`, giá trị phụ thuộc vào thứ tự (lần thứ `t`). Gọi `A_i = {v_i, v_i − d_i, ...}` (các lượt dương). Với tập lượt được chọn `X`, nếu `X ∩ A_i` chứa phần tử thứ `j` mà thiếu phần tử thứ `j' < j` (lớn hơn) thì thay phần tử thứ `j` bằng thứ `j'` không giảm tổng và vẫn không vượt quá `m` lượt. Do đó tối ưu có thể giả sử mỗi `X ∩ A_i` là một **tiền tố** của `A_i`, thực hiện được bằng cách chọn `i` đúng số lần đó. Bài toán trở thành chọn tối đa `m` lượt dương để tổng lớn nhất ⇒ lấy `m` lượt dương lớn nhất (nếu có đủ), vì lượt dương nào cũng làm tổng tăng.

**Tập `m` lượt lớn nhất phù hợp tiền tố.** Nếu có nhiều lượt cùng giá trị ở ngưỡng, ta có quyền chọn trong chúng; có thể chọn sao cho khớp tiền tố vì trong mỗi dãy giá trị giảm **ngặt** (`decay ≥ 1`), nên một dãy có tối đa một lượt bằng `t*`: chọn nó cùng mọi lượt lớn hơn của dãy đó là tiền tố.

**Công thức ngưỡng.** `count(t*+1) < m ≤ count(t*)`. Mọi lượt `≥ t*+1` nằm trong top-`m`; `count(t*) − count(t*+1)` lượt có giá trị đúng `t*`, đủ để bù `m − count(t*+1) ≥ 1`. Tổng = `S(t*+1) + (m − count(t*+1))·t*`.

**Nhị phân đúng.** `count(t)` không tăng theo `t` (nâng ngưỡng chỉ loại bớt lượt), nên tập `{t : count(t) ≥ m}` là một đoạn `[1, t*]`; nhị phân tìm đúng `t*`. Khi `count(1) ≤ m` không có ngưỡng thỏa `count(t) ≥ m` với `m` đủ lớn (hoặc chỉ `t=1` khi bằng), ta lấy hết lượt dương.

**Công thức `count` và `tổng` đúng** vì cấp số cộng giảm đều: số hạng `j` là `v − jd ≥ t` ⇔ `j ≤ (v−t)/d`.

## 11. Phân tích Time/Space

Đặt `n = len(value)`, `V = max(value)`.

| Cách | Time | Space |
|---|---|---|
| Vét cạn chia số lần | `O(C(m+n, n))` | `O(n)` |
| Heap mô phỏng | `O((n + m) log n)` | `O(n)` |
| Nhị phân ngưỡng (nộp) | `O(n log V)` (`≈ 30 n`) | `O(1)` thêm |

Với `n = 10^5`: khoảng `3 · 10^6` lần tính `//` — tính bằng giây, chấp nhận được trong Python.

In [14]:
import time, random
rng = random.Random(9)
n = 100000
value = [rng.randint(1, 10**9) for _ in range(n)]
decay = [rng.randint(1, 10**9) for _ in range(n)]
for m in [1, 10**5, 10**9]:
    t0 = time.time()
    res = Solution().maxTotalValue(value, decay, m)
    print(f"m={m:>10}: {res}  ({time.time()-t0:.2f}s)")
# decay nhỏ -> rất nhiều lượt cho mỗi chỉ số
value2 = [10**9] * n; decay2 = [1] * n
t0 = time.time()
print("value=1e9, decay=1, m=1e9:", Solution().maxTotalValue(value2, decay2, 10**9), f"({time.time()-t0:.2f}s)")

m=         1: 999998593  (0.10s)


m=    100000: 600992839  (0.23s)
m=1000000000: 307411214  (0.02s)


value=1e9, decay=1, m=1e9: 500035049 (0.62s)


## 12. Bộ test

Chiến lược test:

1. 3 ví dụ của đề (assert).
2. Biên: `n = 1`; `m = 1`; mọi `decay` rất lớn (chỉ lượt đầu dương); `decay = 1` (nhiều lượt); `m` lớn hơn số lượt dương; hòa giá trị ở ngưỡng.
3. Phản ví dụ: không bỏ lượt `≤ 0` và sai công thức phần thiếu (phần 13).
4. Đối chiếu vét cạn, heap, ngưỡng và `Solution` trên input ngẫu nhiên nhỏ (seed cố định); với `m` vừa dùng heap thay vét cạn.

In [15]:
sol = Solution()
for args, expected in examples:
    assert sol.maxTotalValue(**args) == expected

assert sol.maxTotalValue([5], [1], 1) == 5
assert sol.maxTotalValue([5], [1], 3) == 5 + 4 + 3
assert sol.maxTotalValue([5], [1], 100) == 5 + 4 + 3 + 2 + 1            # m lớn hơn số lượt dương
assert sol.maxTotalValue([10, 10], [100, 100], 5) == 20                 # chỉ lượt đầu dương
assert sol.maxTotalValue([3, 3, 3], [1, 1, 1], 4) == 3 + 3 + 3 + 2      # hòa ở ngưỡng 2
n1 = 10**9
exact = n1 * n1 - 1 * n1 * (n1 - 1) // 2                                # 1e9 lượt: 1e9, 1e9-1, ..., 1
assert sol.maxTotalValue([10**9], [1], 10**9) == exact % MOD
print("test cố định OK")

test cố định OK


In [16]:
import random
rng = random.Random(1357)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 4)
    value = [rng.randint(1, 12) for _ in range(n)]
    decay = [rng.randint(1, 8) for _ in range(n)]
    m = rng.randint(1, 7)
    e = brute_force(value, decay, m)
    assert heap_greedy(value, decay, m) == e, (value, decay, m)
    assert threshold_search(value, decay, m) == e, (value, decay, m)
    assert Solution().maxTotalValue(value, decay, m) == e % MOD, (value, decay, m)
    cnt += 1
print("đã đối chiếu", cnt, "input ngẫu nhiên nhỏ: tất cả khớp vét cạn")

đã đối chiếu 1500 input ngẫu nhiên nhỏ: tất cả khớp vét cạn


In [17]:
# Input vừa: so heap (mô phỏng) với ngưỡng nhị phân
rng = random.Random(2468)
for _ in range(300):
    n = rng.randint(1, 40)
    value = [rng.randint(1, 10**rng.randint(1, 6)) for _ in range(n)]
    decay = [rng.randint(1, 10**rng.randint(0, 5)) for _ in range(n)]
    m = rng.randint(1, 500)
    e = heap_greedy(value, decay, m)
    assert threshold_search(value, decay, m) == e
    assert Solution().maxTotalValue(value, decay, m) == e % MOD
print("heap == ngưỡng == Solution trên 300 input vừa")

heap == ngưỡng == Solution trên 300 input vừa


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không dừng ở các lượt `<= 0` (ép dùng đủ `m` lần)

**Code sai:** lấy `m` lượt lớn nhất kể cả lượt âm (chạy heap `m` lần không kiểm tra `g <= 0`).
**Vì sao sai:** đề cho phép chọn **tối đa** `m` lần; lượt âm làm giảm tổng. Ví dụ 3 cho `7` chứ không phải số nhỏ hơn.
**Code đúng:** `if g <= 0: break`.

In [18]:
def wrong_no_break(value, decay, m):
    h = [(-v, i) for i, v in enumerate(value)]
    heapq.heapify(h)
    total = 0
    for _ in range(m):
        ng, i = heapq.heappop(h)
        g = -ng
        total += g                                  # SAI: cộng cả lượt âm
        heapq.heappush(h, (-(g - decay[i]), i))
    return total

print("sai :", wrong_no_break(**examples[2][0]), " đúng:", Solution().maxTotalValue(**examples[2][0]))

sai : 0  đúng: 7


### Lỗi 2 — Quên xử lý phần "hòa" ở ngưỡng

**Code sai:** `return total_ge(t*)` (lấy cả các lượt đúng bằng `t*`) hoặc `total_ge(t*+1)` (bỏ phần thiếu).
**Vì sao sai:** `count(t*)` có thể lớn hơn `m`: ta không được lấy hết lượt bằng `t*`; ngược lại bỏ phần thiếu thì thiếu giá trị. Cần `(m − count(t*+1)) · t*`.
**Code đúng:** `S(t*+1) + (m − count(t*+1)) · t*`.

In [19]:
def wrong_take_all_at_threshold(value, decay, m):
    if count_ge(value, decay, 1) <= m:
        return total_ge(value, decay, 1)
    lo, hi = 1, max(value) + 1
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if count_ge(value, decay, mid) >= m: lo = mid
        else: hi = mid
    return total_ge(value, decay, lo)               # SAI: lấy mọi lượt >= t*, có thể nhiều hơn m lượt

print("sai :", wrong_take_all_at_threshold(**examples[0][0]), " đúng:", Solution().maxTotalValue(**examples[0][0]))

sai : 23  đúng: 19


### Lỗi 3 — Sai số hạng đếm: `(v − t) // d` thay vì `(v − t) // d + 1`

**Code sai:** quên `+1` khi đếm.
**Vì sao sai:** số hạng đầu `j = 0` (giá trị `v` chính nó) cũng được tính; `(v−t)//d` đếm khoảng cách chứ không phải số hạng.
**Code đúng:** `(v − t) // d + 1`.

In [20]:
def wrong_count_ge(value, decay, t):
    return sum((v - t) // d for v, d in zip(value, decay) if v >= t)   # SAI: thiếu +1

print("sai :", wrong_count_ge([6, 5, 4], [2, 1, 1], 4), " đúng:", count_ge([6, 5, 4], [2, 1, 1], 4))

sai : 2  đúng: 5


### Lỗi 4 — Dùng `float` cho tổng hoặc mod giữa chừng với phép chia 2

**Code sai:** `c * (c - 1) / 2` (chia thực) hoặc `(c * (c - 1) // 2) % MOD` rồi nhân tiếp.
**Vì sao sai:** `/` cho `float`, mất chính xác với `c ~ 10^9`; còn mod trước khi nhân với `d` rồi trừ cũng được nhưng phải nhất quán. Cách đơn giản nhất: dùng số nguyên Python và `//`, rồi `% MOD` cuối cùng.
**Code đúng:** `d * c * (c - 1) // 2` (tích chẵn nên chia 2 chính xác).

In [21]:
c, d = 10**9, 7
print("float:", d * c * (c - 1) / 2, "| nguyên:", d * c * (c - 1) // 2)
print("chênh lệch do float:", int(d * c * (c - 1) / 2) - d * c * (c - 1) // 2)

float: 3.4999999965e+18 | nguyên: 3499999996500000000
chênh lệch do float: -256


## 14. Mẹo & nhận dạng pattern

- **"Chọn tối đa `m` lần, mỗi lần lấy phần tử lớn nhất hiện có, giá trị giảm sau mỗi lần chọn"** → top-`m` của hợp các dãy giảm.
- `m` lên tới `10^9` mà mô phỏng từng bước là `O(m)` ⇒ cần **nhị phân trên giá trị ngưỡng** thay vì trên số lần.
- Với "đếm số phần tử `≥ t`" trong cấp số cộng dùng `(v − t)//d + 1`; "tổng" dùng `c·v − d·c(c−1)/2`.
- **Xử lý hòa ở ngưỡng**: lấy hết phần `> t*` rồi bù đúng phần còn thiếu bằng giá trị `t*`.
- Nhận ra bài **"k-th largest trong tập ẩn"**: nhị phân trên đáp án + hàm đếm đơn điệu (giống bài *Kth Smallest in Multiplication Table*).
- Chỉ mod ở cuối khi có thể (Python), tránh sai sót với phép chia.

## 15. Biến thể

1. **Bắt buộc đúng `m` lần** (kể cả lượt âm): nhị phân ngưỡng cho phép `t ≤ 0` (Bài 5).
2. **Mỗi chỉ số có giới hạn số lần chọn `cap[i]`:** `count` thành `min(cap, (v−t)//d + 1)`.
3. **Giá trị giảm theo cấp số nhân** (`v·r^t`): hàm đếm đổi thành `log`.
4. **Tìm giá trị lượt thứ `m`** (không cần tổng): chính là `t*`.
5. **Nhiều truy vấn với các `m` khác nhau:** sắp xếp/nhị phân nhiều lần, hoặc cache `count(t)`.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn chia số lần | thử mọi `c_i` với `Σ c_i ≤ m` | rất lớn | `O(n)` | đối chiếu `n, m` rất nhỏ |
| 2. Heap mô phỏng | lấy lượt lớn nhất, đẩy lượt kế tiếp | `O((n+m) log n)` | `O(n)` | `m` nhỏ, đối chiếu |
| 3. Nhị phân ngưỡng | tìm giá trị lớn thứ `m` rồi cộng công thức | `O(n log V)` | `O(1)` | **nộp bài** |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `gains(7, 3)` trả về gì? `count` số lượt `>= 4` của `v = 7, d = 3` là bao nhiêu?

**Bài 2 (tính tay).** `value = [5, 3]`, `decay = [2, 1]`, `m = 3`. Đáp án là gì?

**Bài 3 (điền TODO).** Hoàn thiện `my_count_ge(value, decay, t)` trả về số lượt `>= t`.

**Bài 4 (tự viết lại).** Viết `my_heap(value, decay, m)` bằng heap mô phỏng (có `break` khi lượt `<= 0`).

**Bài 5 (biến thể).** Viết `exactly_m(value, decay, m)`: phải chọn **đúng** `m` lần (có thể lượt âm). Gợi ý: nhị phân ngưỡng với `t` cho phép âm, cận dưới `-max(decay) * m`.

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_count_ge(value, decay, t):
    # TODO: với mỗi (v, d) có v >= t cộng (v - t) // d + 1
    pass

# Bài 4 — TODO
def my_heap(value, decay, m):
    # TODO: heap (-gain, i); lặp m lần; break khi gain <= 0; đẩy gain - decay[i]
    pass

# Bài 5 — TODO
def exactly_m(value, decay, m):
    # TODO: lo = -max(decay) * m (count(lo) >= m), hi = max(value) + 1; nhị phân; công thức ngưỡng như cách 3
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: các giá trị `7, 4, 1, -2...` — chỉ giữ dương.
- Mức 2: `gains(7, 3) = [7, 4, 1]`.
- Mức 3: `>= 4`: `7` và `4` → `(7-4)//3 + 1 = 2`.

**Bài 2.**
- Mức 1: liệt kê các lượt của từng chỉ số.
- Mức 2: chỉ số 0: `5, 3, 1`; chỉ số 1: `3, 2, 1`.
- Mức 3: top 3 là `5, 3, 3` → `11`.

**Bài 3.**
- Mức 1: giống công thức ở 3.2.
- Mức 2: bỏ qua `v < t`.
- Mức 3: `sum((v - t) // d + 1 for v, d in zip(value, decay) if v >= t)`.

**Bài 4.**
- Mức 1: `heapq.heapify` danh sách `(-v, i)`.
- Mức 2: mỗi lần pop lượt lớn nhất; nếu `<= 0` thì dừng.
- Mức 3: đẩy `-(g - decay[i])`.

**Bài 5.**
- Mức 1: giống cách 3 nhưng không có bước "lấy hết lượt dương".
- Mức 2: `t` âm vẫn dùng công thức `(v - t)//d + 1` (đúng vì `v ≥ t` khi `t` đủ nhỏ).
- Mức 3: `lo = -max(decay)*m` luôn có `count(lo) >= m` (mỗi chỉ số riêng đủ `m` lượt).

## 19. Đáp án tham khảo

In [23]:
# Bài 1
assert gains(7, 3) == [7, 4, 1]
assert count_ge([7], [3], 4) == 2
# Bài 2
assert Solution().maxTotalValue([5, 3], [2, 1], 3) == 11 == brute_force([5, 3], [2, 1], 3)

# Bài 3
def my_count_ge_ans(value, decay, t):
    return sum((v - t) // d + 1 for v, d in zip(value, decay) if v >= t)

rng = random.Random(3)
for _ in range(200):
    value = [rng.randint(1, 30) for _ in range(5)]; decay = [rng.randint(1, 9) for _ in range(5)]
    t = rng.randint(1, 35)
    assert my_count_ge_ans(value, decay, t) == count_ge(value, decay, t) == sum(1 for v, d in zip(value, decay) for g in gains(v, d) if g >= t)
print("Bài 1–3 OK")

Bài 1–3 OK


In [24]:
# Bài 4
def my_heap_ans(value, decay, m):
    h = [(-v, i) for i, v in enumerate(value)]
    heapq.heapify(h)
    total = 0
    for _ in range(m):
        ng, i = heapq.heappop(h)
        g = -ng
        if g <= 0:
            break
        total += g
        heapq.heappush(h, (-(g - decay[i]), i))
    return total

# Bài 5: đúng m lần
def exactly_m_ans(value, decay, m):
    def cnt(t):
        return sum((v - t) // d + 1 for v, d in zip(value, decay) if v >= t)
    def tot(t):
        s = 0
        for v, d in zip(value, decay):
            if v >= t:
                c = (v - t) // d + 1
                s += c * v - d * c * (c - 1) // 2
        return s
    lo, hi = -max(decay) * m, max(value) + 1
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if cnt(mid) >= m: lo = mid
        else: hi = mid
    return tot(lo + 1) + (m - cnt(lo + 1)) * lo

rng = random.Random(77)
for _ in range(400):
    n = rng.randint(1, 3)
    value = [rng.randint(1, 10) for _ in range(n)]; decay = [rng.randint(1, 8) for _ in range(n)]
    m = rng.randint(1, 6)
    assert my_heap_ans(value, decay, m) == brute_force(value, decay, m)
    assert exactly_m_ans(value, decay, m) == exact_m_brute(value, decay, m), (value, decay, m)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "top-m của hợp các dãy giảm" bằng nhị phân ngưỡng:**

```python
def stat(t):                                  # (số lượt >= t, tổng các lượt >= t)
    cnt = tot = 0
    for v, d in zip(value, decay):
        if v >= t:
            c = (v - t) // d + 1
            cnt += c
            tot += c * v - d * c * (c - 1) // 2
    return cnt, tot

if stat(1)[0] <= m: return stat(1)[1] % MOD          # lấy hết lượt dương
lo, hi = 1, max(value) + 1                           # count(lo) > m, count(hi) = 0
while hi - lo > 1:
    mid = (lo + hi) // 2
    if stat(mid)[0] >= m: lo = mid
    else: hi = mid
cnt, tot = stat(lo + 1)
return (tot + (m - cnt) * lo) % MOD
```

**Checklist trước khi submit:**

- [ ] Đã đưa về "tổng `m` số dương lớn nhất của hợp các cấp số cộng giảm".
- [ ] Trường hợp `count(1) <= m` được xử lý trước (không ép dùng đủ `m` lần).
- [ ] Số hạng `≥ t`: `(v − t)//d + 1` và chỉ khi `v ≥ t`.
- [ ] Tổng cấp số cộng: `c·v − d·c(c−1)//2` (dùng `//`, không dùng `/`).
- [ ] Xử lý hòa: `S(t*+1) + (m − count(t*+1))·t*`.
- [ ] Biên nhị phân: `lo = 1` (đúng), `hi = max(value) + 1` (sai).
- [ ] `% MOD` ở bước cuối.
- [ ] Đã đối chiếu vét cạn/heap trên input nhỏ ngẫu nhiên.